# Automating Inference with Snowflake ML Jobs

[ML Jobs](https://docs.snowflake.com/en/developer-guide/snowflake-ml/ml-jobs/overview) let you run Python workloads on [Compute Pools](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/working-with-compute-pool) (GPU/CPU containers) — ideal for heavy inference or training that exceeds warehouse capabilities.

Submission methods: [`submit_file`](https://docs.snowflake.com/en/developer-guide/snowflake-ml/ml-jobs/overview#run-a-python-file-as-a-snowflake-ml-job) | [`submit_directory`](https://docs.snowflake.com/en/developer-guide/snowflake-ml/ml-jobs/overview#run-a-python-file-as-a-snowflake-ml-job) | [`@remote` decorator](https://docs.snowflake.com/en/developer-guide/snowflake-ml/ml-jobs/overview#run-a-python-function-as-a-snowflake-ml-job)

Below we wrap inference in a Stored Procedure that submits an ML Job, then schedule it with a Task.

In [ ]:
%%sql -r dir_proc_result
CREATE OR REPLACE PROCEDURE ML_LAB.DATA.directory_inference_ml_job() 
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = '3.10'
PACKAGES = ('snowflake-ml-python', 'snowflake-snowpark-python')
HANDLER = 'run'
AS
$$

from snowflake.ml.jobs import submit_directory

def run(session):
    ml_job_dr = submit_directory(
        ".", 
        "CPU_POOL",
        entrypoint="inference.py",
        stage_name="ML_LAB.DATA.payload_stage",
        session=session,
    )
    ml_job_dr.wait()
    return f"Job {ml_job_dr.id} finished with status: {ml_job_dr.status}"
$$;

In [ ]:
CREATE OR REPLACE TASK directory_inference_ml_job
  WAREHOUSE = COMPUTE_WH
  SCHEDULE = 'USING CRON 0 7 * * * UTC'
AS
  CALL trigger_iris_ml_job();

ALTER TASK directory_inference_ml_job RESUME;

# Optional - Schedule a single python file

In a stored procedure, the code runs in an isolated Snowflake execution environment — not in your workspace. The workspace filesystem (snow://workspace/...) is only accessible from notebook cells running interactively, not from within a stored procedure or ML Job container.

So when the stored procedure calls submit_file, it needs to reference a file on a Snowflake stage because that's the only shared storage the ML Job container can access at runtime.

In [ ]:
%%sql -r copy_result
COPY FILES INTO @ML_LAB.DATA.payload_stage
FROM 'snow://workspace/USER$.PUBLIC."Python-Code-Automation-In-Snowflake"/versions/live/'
FILES=('inference.py');

# Create a store procedure wrapping ML job

In [ ]:
CREATE OR REPLACE PROCEDURE ML_LAB.DATA.single_file_inference_ml_job() 
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = '3.10'
PACKAGES = ('snowflake-ml-python', 'snowflake-snowpark-python')
HANDLER = 'run'
AS
$$

from snowflake.ml.jobs import submit_file

def run(session):
    ml_job = submit_file(
        "@ML_LAB.DATA.payload_stage/inference.py", 
        "CPU_POOL",
        stage_name="ML_LAB.DATA.payload_stage",
        session=session,
    )
    ml_job.wait()
    return f"Job {ml_job.id} finished with status: {ml_job.status}"
$$;

In [ ]:
CREATE OR REPLACE TASK single_file_ml_job_task
  WAREHOUSE = COMPUTE_WH
  SCHEDULE = 'USING CRON 0 7 * * * UTC'
AS
  CALL single_file_inference_ml_job();

ALTER TASK single_file_ml_job_task RESUME;